# 真实本地模型：EigenScore 与 Effective Rank

从本地因果模型提取真实生成 token 的隐藏状态，运行两个实验性表示 scorer。

## 真实集成前提

安装：`python3 -m pip install -e ".[whitebox]"`。开启隐藏状态会增加资源消耗。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `LOCAL_CAUSAL_MODEL: str`：本地 causal LM 目录。
- `hidden_state_layer: int = -1`：提取的层。
- `hidden_state_pooling: str = 'none'`：保留每个生成 token 的向量。
- `alpha: float = 1e-3`：EigenScore 特征值正则项。

## 步骤 1：配置真实后端

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from hallucination.detection.backends.hf_whitebox import HuggingFaceWhiteBoxBackend
LOCAL_CAUSAL_MODEL = "/absolute/path/to/your/causal-lm"
DEVICE = "cpu"
MAX_NEW_TOKENS = 48
TOP_K = 5
if not Path(LOCAL_CAUSAL_MODEL).is_dir():
    raise ValueError("Set LOCAL_CAUSAL_MODEL to an existing local model directory.")
backend = HuggingFaceWhiteBoxBackend.from_pretrained(LOCAL_CAUSAL_MODEL, device=DEVICE)

## 步骤 2：提取隐藏状态并评分

In [ ]:
from hallucination.detection.whitebox.effective_rank import EffectiveRankScorer
from hallucination.detection.whitebox.eigenscore import EigenScoreScorer
# 先多次生成，再用 backend.generate_hidden_states(..., n_samples=5) 得到 [5, hidden]。
generation = backend.generate_with_scores("请用一句中文说明蓝鲸主要吃什么。", max_new_tokens=MAX_NEW_TOKENS, top_k=TOP_K, include_hidden_states=True, hidden_state_layer=-1, hidden_state_pooling="none", do_sample=False)
example = generation.to_detection_example()
print(getattr(example.hidden_states, "shape", None))
for scorer in [EigenScoreScorer(alpha=1e-3), EffectiveRankScorer()]:
    print(scorer.score(example))

## 结果解读与限制

层、池化、模型架构和 token 数都会改变数值。这两个方法仍是实验性实现，必须在固定协议和独立验证集上验证。